# Manufacturing Production Data Quality Assessment

**Portfolio Simulation — WI-DA-001 Technical Implementation**

This notebook implements a controlled, traceable data-quality assessment workflow for a simulated manufacturing production dataset.

The workflow separates:

1. **Schema validation**
2. **Domain validation**
3. **Cross-field business rules**
4. **Statistical review flags**

> **Data integrity principle:** Statistical extremity alone is not treated as proof that a record is invalid. Potential anomalies are flagged for review, not automatically deleted.

## Workflow Objectives

- Preserve the original source dataset.
- Load a working copy for validation.
- Apply controlled rules from `validation_rules.yaml`.
- Generate requirement-level validation evidence.
- Produce a detailed issue log.
- Determine dataset release eligibility.
- Compare detected issues with a controlled development defect manifest.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import yaml

ROOT = Path("..")
DATA_PATH = ROOT / "data" / "working" / "production_batch_2026_001_working.csv"
RULES_PATH = ROOT / "config" / "validation_rules.yaml"
DICTIONARY_PATH = ROOT / "data" / "reference" / "production_data_dictionary.csv"

df_source = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False)
df = df_source.copy()  # approved working-copy pattern for this portfolio simulation

with open(RULES_PATH, "r", encoding="utf-8") as f:
    rules = yaml.safe_load(f)

data_dictionary = pd.read_csv(DICTIONARY_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

## Important Control

This notebook does **not** automatically delete duplicates, impute missing values, replace outliers, or overwrite the source file. Validation identifies conditions requiring resolution; it does not silently modify production data.

In [ ]:
required_identifiers = ["Record_ID", "Facility_ID", "Machine_ID", "Batch_ID", "Timestamp"]
missing_identifier_summary = {
    col: (df[col].astype(str).str.strip() == "").sum()
    for col in required_identifiers
}
missing_identifier_summary

In [ ]:
business_key = ["Facility_ID", "Batch_ID", "Machine_ID", "Timestamp"]
duplicate_mask = df.duplicated(subset=business_key, keep=False)
potential_duplicates = df.loc[duplicate_mask, ["Record_ID"] + business_key]

print(f"Potential duplicate records: {len(potential_duplicates)}")
potential_duplicates.head(10)

In [ ]:
numeric_columns = [
    "Temperature_C", "Pressure_kPa", "Line_Speed_UPM", "Cycle_Time_sec",
    "Energy_kWh", "Units_Produced", "Defect_Count", "Downtime_Min"
]

numeric_view = {
    col: pd.to_numeric(df[col].replace("", np.nan), errors="coerce")
    for col in numeric_columns
}

type_failures = {
    col: int((df[col].replace("", np.nan).notna() & numeric_view[col].isna()).sum())
    for col in numeric_columns
}
type_failures

In [ ]:
temperature = numeric_view["Temperature_C"]
temperature_range_failures = df.loc[
    temperature.notna() & ~temperature.between(60, 100),
    ["Record_ID", "Temperature_C"]
]

print(f"Temperature hard-range failures: {len(temperature_range_failures)}")
temperature_range_failures.head()

In [ ]:
units = numeric_view["Units_Produced"]
defects = numeric_view["Defect_Count"]

cross_field_failures = df.loc[
    units.notna() & defects.notna() & (defects > units),
    ["Record_ID", "Units_Produced", "Defect_Count"]
]

print(f"Defect_Count > Units_Produced failures: {len(cross_field_failures)}")
cross_field_failures.head()

In [ ]:
valid_temp = temperature.notna() & temperature.between(60, 100)
temp_values = temperature[valid_temp]

median = temp_values.median()
mad = (temp_values - median).abs().median()
robust_z = pd.Series(np.nan, index=df.index)

if mad > 0:
    robust_z.loc[valid_temp] = 0.6745 * (temp_values - median) / mad

review_flags = df.loc[
    valid_temp & (robust_z.abs() > 3.2),
    ["Record_ID", "Temperature_C"]
].copy()
review_flags["Robust_Z"] = robust_z.loc[review_flags.index].round(2)

print(f"Temperature review flags: {len(review_flags)}")
review_flags.head(10)

## Interpretation

Hard validity limits and statistical review flags serve different purposes:

- A **hard-range failure** violates a controlled requirement.
- A **statistical anomaly** is unusual relative to the observed distribution but may still be valid manufacturing data.

The workflow therefore preserves anomalous observations unless another approved requirement establishes that the record is invalid.

## Controlled Outputs

The full implementation generates:

- `validation_summary.csv`
- `validation_issues.csv`
- `release_report.csv`
- `defect_detection_test_report.csv`

These outputs provide the evidence layer required for the accompanying Work Instruction.